# Analyzing Historical Stock/Revenue Data and Building a Dashboard

IBM Data Analyst Professional Certificate — Final Assignment

This notebook covers the six practical exercises: Tesla stock data, Tesla revenue data, GameStop stock data, GameStop revenue data, and the two stock graphs.

**Before running:** Run the notebook in Jupyter or Google Colab with internet access. If needed, install the dependencies in the first code cell. Data providers can change their page layouts or restrict automated requests, so the revenue-scraping cells include checks and may need a small adjustment if a source changes.

In [ ]:
# Install dependencies if they are not already installed.
# In Google Colab/Jupyter, uncomment the next line if imports fail.
# %pip install yfinance pandas requests beautifulsoup4 lxml plotly

## Exercise 1 — Use yfinance to extract Tesla stock data

In [ ]:
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from IPython.display import display

# Download Tesla historical stock data
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")

# Reset the index and display the first five rows
tesla_data.reset_index(inplace=True)
display(tesla_data.head())

## Exercise 2 — Web scrape Tesla revenue data

In [ ]:
def scrape_company_revenue(url):
    """Extract a revenue table from a public webpage when a suitable HTML table is available."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Prefer tables with a Date/Revenue header, regardless of table class.
    for table in soup.find_all("table"):
        rows = table.find_all("tr")
        if not rows:
            continue
        headers_row = [cell.get_text(" ", strip=True).lower()
                       for cell in rows[0].find_all(["th", "td"])]
        if any("date" in h for h in headers_row) and any("revenue" in h for h in headers_row):
            records = []
            for row in rows[1:]:
                values = [cell.get_text(" ", strip=True)
                          for cell in row.find_all(["th", "td"])]
                if len(values) >= 2:
                    records.append(values[:2])
            if records:
                df = pd.DataFrame(records, columns=["Date", "Revenue"])
                df["Revenue"] = (df["Revenue"].astype(str)
                                 .str.replace("$", "", regex=False)
                                 .str.replace(",", "", regex=False))
                df = df[df["Revenue"].str.strip().ne("")]
                return df.reset_index(drop=True)

    # Fallback: try pandas' HTML-table parser and select a table with revenue columns.
    tables = pd.read_html(response.text)
    for table in tables:
        cols = [str(c).lower() for c in table.columns]
        if any("date" in c for c in cols) and any("revenue" in c for c in cols):
            table.columns = ["Date" if "date" in str(c).lower()
                             else "Revenue" if "revenue" in str(c).lower()
                             else str(c) for c in table.columns]
            return table[["Date", "Revenue"]].dropna().reset_index(drop=True)
    raise ValueError("Could not find a Date/Revenue table on the source webpage.")

tesla_revenue_url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
try:
    tesla_revenue = scrape_company_revenue(tesla_revenue_url)
    display(tesla_revenue.tail())
except Exception as exc:
    print("Tesla revenue scraping was blocked or the webpage layout changed.")
    print("Error:", exc)
    print("Try opening the source in a browser or use the course-provided data source.")
    tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

## Exercise 3 — Use yfinance to extract GameStop stock data

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
display(gme_data.head())

## Exercise 4 — Web scrape GameStop revenue data

In [ ]:
gme_revenue_url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
try:
    gme_revenue = scrape_company_revenue(gme_revenue_url)
    display(gme_revenue.tail())
except Exception as exc:
    print("GameStop revenue scraping was blocked or the webpage layout changed.")
    print("Error:", exc)
    print("Try opening the source in a browser or use the course-provided data source.")
    gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

## Exercise 5 — Plot Tesla stock graph

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def make_graph(stock_data, revenue_data, stock_name):
    """Plot historical closing price and revenue, using separate axes."""
    stock = stock_data.copy()
    if "Date" not in stock.columns:
        stock = stock.reset_index()
    stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce")
    stock = stock.dropna(subset=["Date"])

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    if "Close" in stock.columns:
        fig.add_trace(
            go.Scatter(x=stock["Date"], y=stock["Close"],
                       name="Share Price", mode="lines"),
            secondary_y=False
        )

    revenue = revenue_data.copy()
    if not revenue.empty and {"Date", "Revenue"}.issubset(revenue.columns):
        revenue["Date"] = pd.to_datetime(revenue["Date"], errors="coerce")
        revenue["Revenue"] = (revenue["Revenue"].astype(str)
                              .str.replace("$", "", regex=False)
                              .str.replace(",", "", regex=False))
        revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="coerce")
        revenue = revenue.dropna(subset=["Date", "Revenue"])
        fig.add_trace(
            go.Scatter(x=revenue["Date"], y=revenue["Revenue"],
                       name="Revenue", mode="lines+markers"),
            secondary_y=True
        )

    fig.update_layout(
        title=f"{stock_name} Stock Price and Revenue",
        xaxis_title="Date",
        template="plotly_white",
        legend_title="Metric"
    )
    fig.update_yaxes(title_text="Share Price (USD)", secondary_y=False)
    fig.update_yaxes(title_text="Revenue (USD)", secondary_y=True)
    fig.show()

make_graph(tesla_data, tesla_revenue, "Tesla")

## Exercise 6 — Plot GameStop stock graph

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Submission checklist

1. Run every code cell from top to bottom.
2. Confirm the stock-data tables and both graphs display.
3. If a revenue source blocks scraping, use the exact data-source approach specified by your Coursera lab and rerun the affected cells.
4. Save the notebook with outputs included (`.ipynb`).
5. Capture the code and output requested for each Coursera question. The Coursera page shown in your screenshot has six screenshot fields (Questions 1–6), plus a GitHub URL field.
6. Upload the notebook to a GitHub repository you control and paste that repository/file URL into the assignment.

Do not submit screenshots until the notebook has actually run and the visible results match your own execution.